<a href="https://colab.research.google.com/github/Sachin-Baranwal1/Machine-Learning-Projects/blob/main/Fake_News_Prediction_using_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. id: unique id for a news article
2. title: the title of a news article
3. author: author of the news article
4. text: the text of the article, could be incomplete
5. label: a marks whether the news article is real or fake

1 --> Fake News

0 --> real News

Importing the dependencies

In [4]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [5]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [6]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [8]:
news_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.csv')

In [7]:
news_data.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [18]:
# Counting the missing values
news_data.isnull().sum()

,0
id,0
title,0
author,0
text,0
label,0


In [17]:
# fill the null value with a empty string
news_data = news_data.fillna(' ')

In [19]:
# Merge the author name and news title
news_data['content'] = news_data['author'] + ' '+news_data['title']

In [29]:
news_data['content'].head()

,content
0,Darrell Lucus House Dem Aide: We Didn’t Even S...
1,"Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo..."
2,Consortiumnews.com Why the Truth Might Get You...
3,Jessica Purkiss 15 Civilians Killed In Single ...
4,Howard Portnoy Iranian woman jailed for fictio...


In [23]:
X = news_data.drop(columns='label' ,axis=1)
Y = news_data['label']

In [24]:
print(X)
print(Y)

          id  ...                                            content
0          0  ...  Darrell Lucus House Dem Aide: We Didn’t Even S...
1          1  ...  Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo...
2          2  ...  Consortiumnews.com Why the Truth Might Get You...
3          3  ...  Jessica Purkiss 15 Civilians Killed In Single ...
4          4  ...  Howard Portnoy Iranian woman jailed for fictio...
...      ...  ...                                                ...
20795  20795  ...  Jerome Hudson Rapper T.I.: Trump a ’Poster Chi...
20796  20796  ...  Benjamin Hoffman N.F.L. Playoffs: Schedule, Ma...
20797  20797  ...  Michael J. de la Merced and Rachel Abrams Macy...
20798  20798  ...  Alex Ansary NATO, Russia To Hold Parallel Exer...
20799  20799  ...            David Swanson What Keeps the F-35 Alive

[20800 rows x 5 columns]
0        1
1        0
2        1
3        1
4        1
        ..
20795    0
20796    0
20797    0
20798    1
20799    1
Name: label, Length: 2080

  Stemming:

  It is the process of reducing a word to its Root word

  example: actor,actress,acting --> act

In [25]:
port_stem = PorterStemmer()

In [42]:
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]',' ',content)
  print(stemmed_content)
  stemmed_content = stemmed_content.lower()
  print(stemmed_content)
  stemmed_content = stemmed_content.split()
  print(stemmed_content)
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  print(stemmed_content)
  stemmed_content = ' '.join(stemmed_content)
  return stemmed_content

In [44]:
news_data['content'] = news_data['content'].apply(stemming)

Streaming output truncated to the last 5000 lines.
Dennis Jackson  
dennis jackson  
['dennis', 'jackson']
['denni', 'jackson']
Ian Hanchett Jane Fonda  I Won t Call Trump  By His Name     I Call Him the Predator In Chief    Breitbart
ian hanchett jane fonda  i won t call trump  by his name     i call him the predator in chief    breitbart
['ian', 'hanchett', 'jane', 'fonda', 'i', 'won', 't', 'call', 'trump', 'by', 'his', 'name', 'i', 'call', 'him', 'the', 'predator', 'in', 'chief', 'breitbart']
['ian', 'hanchett', 'jane', 'fonda', 'call', 'trump', 'name', 'call', 'predat', 'chief', 'breitbart']
Jack Burns Not Just Hillary  Entire Obama Administration Exposed for Using Private Email to Avoid FOIA Requests
jack burns not just hillary  entire obama administration exposed for using private email to avoid foia requests
['jack', 'burns', 'not', 'just', 'hillary', 'entire', 'obama', 'administration', 'exposed', 'for', 'using', 'private', 'email', 'to', 'avoid', 'foia', 'requests']
['jack', '

In [45]:
print(news_data)

          id  ...                                            content
0          0  ...  darrel lucu hous dem aid even see comey letter...
1          1  ...  daniel j flynn flynn hillari clinton big woman...
2          2  ...             consortiumnew com truth might get fire
3          3  ...  jessica purkiss civilian kill singl us airstri...
4          4  ...  howard portnoy iranian woman jail fiction unpu...
...      ...  ...                                                ...
20795  20795  ...  jerom hudson rapper trump poster child white s...
20796  20796  ...  benjamin hoffman n f l playoff schedul matchup...
20797  20797  ...  michael j de la merc rachel abram maci said re...
20798  20798  ...  alex ansari nato russia hold parallel exercis ...
20799  20799  ...                          david swanson keep f aliv

[20800 rows x 6 columns]


In [46]:
# separating the data and label
X =  news_data['content'].values
Y =  news_data['label'].values

In [47]:
print(X)
print(Y)

['darrel lucu hous dem aid even see comey letter jason chaffetz tweet'
 'daniel j flynn flynn hillari clinton big woman campu breitbart'
 'consortiumnew com truth might get fire' ...
 'michael j de la merc rachel abram maci said receiv takeov approach hudson bay new york time'
 'alex ansari nato russia hold parallel exercis balkan'
 'david swanson keep f aliv']
[1 0 1 ... 0 1 1]


In [49]:
# converting the textual data to numerical data
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(X)

In [50]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 210687 stored elements and shape (20800, 17128)>
  Coords	Values
  (0, 3600)	0.3598939188262559
  (0, 8909)	0.3635963806326075
  (0, 7005)	0.21874169089359144
  (0, 3792)	0.2705332480845492
  (0, 267)	0.27010124977708766
  (0, 4973)	0.233316966909351
  (0, 13473)	0.2565896679337957
  (0, 2959)	0.2468450128533713
  (0, 8630)	0.29212514087043684
  (0, 7692)	0.24785219520671603
  (0, 2483)	0.3676519686797209
  (0, 15686)	0.28485063562728646
  (1, 3568)	0.26373768806048464
  (1, 5503)	0.7143299355715573
  (1, 6816)	0.1904660198296849
  (1, 2813)	0.19094574062359204
  (1, 1497)	0.2939891562094648
  (1, 16799)	0.30071745655510157
  (1, 2223)	0.3827320386859759
  (1, 1894)	0.15521974226349364
  (2, 3103)	0.46097489583229645
  (2, 2943)	0.3179886800654691
  (2, 15611)	0.41544962664721613
  (2, 9620)	0.49351492943649944
  (2, 5968)	0.3474613386728292
  :	:
  (20797, 9588)	0.17455348025522197
  (20797, 7042)	0.21799048897828685
  (207

In [57]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y, test_size=0.2,stratify=Y,random_state=2)

Splitting the dataset to training & testing data

In [58]:
print(X.shape,X_train.shape,X_test.shape)

(20800, 17128) (16640, 17128) (4160, 17128)


Training the Model : Logistic Regression

In [59]:
model = LogisticRegression()

In [60]:
model.fit(X_train,Y_train)

LogisticRegression()

Accuracy check of data

In [62]:
#accuracy score on the training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction,Y_train)

In [68]:
print('Accuracy score of training data : ', training_data_accuracy)

Accuracy score of training data :  0.9863581730769231


In [69]:
#accuracy score on the testing data
X_test_prediction = model.predict(X_test)
testing_data_accuracy = accuracy_score(X_test_prediction,Y_test)

In [70]:
print('Accuracy score of training data : ', testing_data_accuracy)

Accuracy score of training data :  0.9790865384615385


Making a Predictive System

In [97]:
column_no = 136 # this is column no/index of training dataset
X_new = X_test[column_no]

prediction = model.predict(X_new)
if prediction == 1:
  print('The Predicted News is --> Fake')
else:
  print('The Predicted News is --> Real')
print('The Real output of which should come is : ',
      'Fake' if Y_test[column_no] == 1 else 'Real' )

The Predicted News is --> Real
The Real output of which should come is :  Real
